# 🧠 Agentic RAG: Interactive Pipeline Walkthrough

This notebook provides a step-by-step interactive demonstration of the **Agentic Retrieval-Augmented Generation** pipeline:

1. **Ingestion & Connectors** (`pipeline/ingestion.py`)
2. **Preprocessing & Normalization** (`pipeline/preprocessing.py`)
3. **Recursive Chunking & Metadata Enrichment** (`pipeline/chunking.py`)
4. **Dense Embeddings & Persistent Vector DB** (`pipeline/embeddings.py`)
5. **Sparse Keyword Inverted Index** (`pipeline/bm25_index.py`)
6. **Hybrid Retrieval (RRF) & Cross-Encoder Reranking** (`pipeline/retrieval.py`)
7. **Grounded Generation & Citation Mapping** (`pipeline/generator.py`)
8. **End-to-End Orchestrator** (`pipeline/orchestrator.py`)

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config import load_config
from pipeline import IngestionPipeline, RAGPipeline, QueryEngine
from evaluation.retrieval_eval import RetrievalEvaluator
from evaluation.generation_eval import GenerationEvaluator

config = load_config()
print("Config loaded successfully!")

## 1. Execute Ingestion Pipeline
Reads all documents from `data/raw/` and `documents/`, cleans them, segments into overlapping chunks, and builds both Vector and BM25 indices.

In [ ]:
ingestion = IngestionPipeline(config, project_root=project_root)
summary = ingestion.run()

print(f"Ingested Documents : {summary.total_documents}")
print(f"Total Chunks       : {summary.total_chunks}")
print(f"Total Tokens       : {summary.total_tokens}")
print(f"Ingestion Time     : {summary.duration_seconds}s")

## 2. Inspect Generated Chunks and Enriched Features

In [ ]:
import json
chunks_path = project_root / "data" / "processed" / "chunks.json"
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

print(f"Showing first chunk out of {len(chunks_data)}:\n")
sample_chunk = chunks_data[0]
print("Chunk ID:", sample_chunk["chunk_id"])
print("Document:", sample_chunk["doc_id"])
print("Extracted Keywords:", sample_chunk["metadata"].get("keywords", []))
print("\nText Snippet:\n", sample_chunk["text"][:200], "...")

## 3. Query Understanding & Transformation

In [ ]:
sample_query = "What is the annual home office stipend and what are the core work hours?"
processed = QueryEngine.process_query(sample_query)

print("Raw Query:", processed["raw_query"])
print("Cleaned Query:", processed["cleaned_query"])
print("Detected Intent:", processed["intent"])
print("Extracted Keywords:", processed["keywords"])

## 4. End-to-End RAG Query with Grounded Citations

In [ ]:
rag = RAGPipeline(config, project_root=project_root)
response = rag.query(sample_query)

print("="*60)
print("QUERY:", response.query)
print("="*60)
print("\nGROUNDED ANSWER:\n", response.answer)
print("\nCONFIDENCE SCORE:", response.confidence_score)
print("LATENCY:", f"{response.latency_ms} ms")
print("\nSOURCES / CITATIONS:")
for c in response.citations:
    print(f"  - [{c.chunk_id}] {c.doc_id} (Score: {c.score:.4f})")
    print(f"    Preview: {c.snippet}")

## 5. Technical Architecture Query Test

In [ ]:
tech_query = "How does Reciprocal Rank Fusion improve hybrid retrieval?"
tech_response = rag.query(tech_query)

print("Tech Answer:\n", tech_response.answer)
print("\nSources:", tech_response.sources)